# 000 — Constants & Tables

Everything hardcoded in `ITU_CERTIFIED_Battle_of_the_Beams_Calc2_v9_1_1.xlsx`
lives here. The remaining chapters import this module (`common.py`)
and reference these tables by name.

The mapping from spreadsheet sheets to Python objects:

| Sheet | Object |
|---|---|
| `const`        | module-level constants (`EPS_0`, `MU_0`, `C_VAC`, `R_EARTH`, `K_REFRAC`, `T_SYS`, `RX_NF_DB`, `RX_BW_HZ`) |
| `ground`       | `GROUND` dict (sigma, eps_r per ground type) |
| `stations`     | `STATIONS` dict (Kleve, Stollberg, Greny, Beaumont-Hague) |
| `targets`      | `TARGETS` dict (Midlands cities + Telefunken sea tests) |
| `paths_refs`   | reproduced inline below; haversine distance is computed from `STATIONS` + `TARGETS` |
| `Squint Sandbox` | `SANDBOX` dict (squint 5°, W = 99 m, H = 20 m) |


In [1]:
import math, pandas as pd
from IPython.display import Markdown, display
import common as c


## Physical constants (`const` sheet)


In [2]:
pd.DataFrame([
    ("eps_0",         c.EPS_0,         "F/m"),
    ("mu_0",          c.MU_0,          "H/m"),
    ("c (= 1/sqrt(eps_0 mu_0))", c.C_VAC, "m/s"),
    ("pi",            math.pi,         ""),
    ("k_Boltzmann",   c.K_BOLTZ,       "J/K"),
    ("R_Earth",       c.R_EARTH,       "m"),
    ("k (4/3 refraction)", c.K_REFRAC, ""),
    ("R_effective = k*R_Earth", c.R_EFFECTIVE, "m"),
    ("T_sys",         c.T_SYS,         "K"),
    ("RX noise figure NF", c.RX_NF_DB, "dB"),
    ("RX bandwidth B",     c.RX_BW_HZ, "Hz"),
    ("eta_0",         c.ETA_0,         "ohm"),
], columns=["Symbol","Value","Unit"]).style.format({"Value": "{:.6g}"})


,Symbol,Value,Unit
0,eps_0,8.85419e-12,F/m
1,mu_0,1.25664e-06,H/m
2,c (= 1/sqrt(eps_0 mu_0)),2.99792e+08,m/s
3,pi,3.14159,
4,k_Boltzmann,1.38065e-23,J/K
5,R_Earth,6.371e+06,m
6,k (4/3 refraction),1.33333,
7,R_effective = k*R_Earth,8.49467e+06,m
8,T_sys,290,K
9,RX noise figure NF,10,dB


## Ground electrical properties (`ground` sheet)

These feed the Fock $\beta$ via ITU-R P.526-16 Eqs. 16 / 16a and the
Sommerfeld-Norton complex permittivity $n^2 = \varepsilon_r - jx$ with
$x = 18\,000\,\sigma / f_{\text{MHz}}$.


In [3]:
pd.DataFrame([
    (k, v["sigma"], v["eps_r"], v["note"])
    for k, v in c.GROUND.items()
], columns=["ground_type","sigma (S/m)","eps_r","notes"])


,ground_type,sigma (S/m),eps_r,notes
0,land,0.005,15,ITU-R P.527 class 3 typical
1,sea,5.000,70,warm/typical North Sea
2,wet,0.020,30,wet soil
3,dry,0.001,4,dry soil


## Stations (`stations` sheet)


In [4]:
pd.DataFrame([
    dict(name=k, **v) for k, v in c.STATIONS.items()
])


,name,short_id,lat_deg,lon_deg,terrain_elev_m,frame_height_m,h_tx_m,W_m,H_m,freq_MHz,pol,Ptx_W,ground,squint_deg
0,Kleve,Kn-4,51.7886,6.1031,83,28,111,99,29,31.5,vertical,3000,land,5
1,Stollberg,Kn-2,54.6436,8.9447,44,28,72,99,29,31.5,vertical,3000,sea,5
2,Greny,Kn-7,49.9467,1.2900,134,28,162,99,29,31.5,vertical,3000,sea,5
3,Beaumont-Hague,Kn-9,49.6733,-1.8525,169,28,197,99,29,31.5,vertical,3000,sea,5


## Targets (`targets` sheet)


In [5]:
pd.DataFrame([
    dict(name=k, **v) for k, v in c.TARGETS.items()
])


,name,lat,lon,rx_alt_m,note
0,Spalding,52.787000,-0.153000,6000,Bufton 21 Jun 1940 Kleve beam
1,Beeston,52.927000,-1.215000,6000,Bufton 21 Jun 1940 Stollberg beam
2,Derby,52.922000,-1.475000,6000,Operational target Rolls-Royce
3,Birmingham,52.486200,-1.890400,6000,Operational target
4,Retford,53.400000,-1.000000,6000,Enigma intercept 5 Jun 1940
5,London,51.507400,-0.127800,6000,Operational target
6,Liverpool,53.408400,-2.991600,6000,Stollberg target
7,Cardiff,51.481600,-3.179100,6000,Beaumont-Hague target
8,Plymouth,50.375500,-4.142700,6000,Beaumont-Hague target
9,TF 400 km,54.589878,2.518619,4000,Telefunken Sep 1939 FuBl1 rod


## Path-distance table (`paths_refs` sheet, recomputed)

Distances are recomputed by haversine from the station lat/lon and
target lat/lon so the table stays in sync with the source data.


In [6]:
rows = []
TX = {
    'Kleve': ['Spalding','Retford','Derby','Birmingham','TF 400 km','TF 500 km','TF 700 km','TF 800 km','TF 1000 km'],
    'Stollberg': ['Beeston','Derby','Birmingham','Liverpool','TF 400 km','TF 500 km','TF 700 km','TF 800 km','TF 1000 km'],
    'Greny': ['London'],
    'Beaumont-Hague': ['London','Cardiff','Plymouth'],
}
for tx, targets in TX.items():
    s = c.STATIONS[tx]
    for tgt in targets:
        t = c.TARGETS[tgt]
        d_km = c.great_circle_m(s['lat_deg'], s['lon_deg'], t['lat'], t['lon']) / 1000
        ground = 'sea' if tgt.startswith('TF') else s['ground']
        rows.append((tx, tgt, round(d_km, 1), ground))
pd.DataFrame(rows, columns=['tx_station','target','d_km','ground'])


,tx_station,target,d_km,ground
0,Kleve,Spalding,439.6,land
1,Kleve,Retford,511.9,land
2,Kleve,Derby,529.6,land
3,Kleve,Birmingham,550.7,land
4,Kleve,TF 400 km,392.4,sea
5,Kleve,TF 500 km,449.9,sea
6,Kleve,TF 700 km,606.9,sea
7,Kleve,TF 800 km,696.1,sea
8,Kleve,TF 1000 km,874.2,sea
9,Stollberg,Beeston,693.5,sea


## Squint Sandbox defaults

The Squint Sandbox sheet in the workbook is where the British-measured
400 to 500 yard equisignal corridor at Spalding is calibrated. The
default values below recover the operational corridor:

| Input | Default | Symbol |
|---|---|---|
| Squint angle | 5° | $\theta$ |
| Aperture width | 99 m | $W$ |
| Aperture height | 20 m | $H$ |

Sources: Trenkle 1979 p. 67 for the 99 m sub-array array width;
Telefunken 5° squint per Bauer 2004 p. 12. The 20 m aperture-H
default is the Squint Sandbox sheet cell B18.


In [7]:
pd.DataFrame([
    ("Squint theta (deg)", c.SANDBOX['squint_deg']),
    ("Aperture W (m)",     c.SANDBOX['W_m']),
    ("Aperture H (m)",     c.SANDBOX['H_m']),
], columns=["Sandbox input","Default"])


,Sandbox input,Default
0,Squint theta (deg),5.0
1,Aperture W (m),99.0
2,Aperture H (m),20.0


## Derived constants at Knickebein parameters

These follow algebraically from the table above and the constants and
are quoted everywhere downstream.


In [8]:
f_MHz = 31.5
W, H = 99.0, 29.0           # ITU sheet uses 29 m for aperture H in stations
lam = c.freq_to_wavelen(f_MHz)
k_wave = c.wavenumber(f_MHz)
G_tx_dBi = c.aperture_gain_dBi(W, H, f_MHz)
P_tx_dBW = 10*math.log10(3000)
N_dBW = c.noise_floor_dBW(f_MHz)
Fa_dB = c.galactic_Fa_dB(f_MHz)
thermal_dBW = 10*math.log10(c.K_BOLTZ*c.T_SYS*c.RX_BW_HZ)
display(Markdown(rf'''
- $\lambda = c/f$ = **{lam:.4f} m**
- $k = 2\pi/\lambda$ = **{k_wave:.4f} rad/m**
- Aperture area $A = W H$ = {W*H:.0f} m²
- Aperture directivity $G_{{tx}} = 4\pi A/\lambda^2$ = **{G_tx_dBi:.2f} dBi**
- $P_{{tx}} = 10 \log_{{10}}(3000)$ = **{P_tx_dBW:.3f} dBW**
- Thermal $kTB$ at 290 K, 500 Hz = **{thermal_dBW:.3f} dBW**
- Galactic $F_a = 52 - 23 \log_{{10}}(f_{{MHz}})$ at 31.5 MHz = **{Fa_dB:.2f} dB**
- Noise floor $N = kTB \cdot \max(NF, F_a)$ = **{N_dBW:.3f} dBW**, equivalent to
  **{c.voltage_50ohm_uV(N_dBW)*1000:.2f} nV** at the 50 ohm input.
'''))



- $\lambda = c/f$ = **9.5172 m**
- $k = 2\pi/\lambda$ = **0.6602 rad/m**
- Aperture area $A = W H$ = 2871 m²
- Aperture directivity $G_{tx} = 4\pi A/\lambda^2$ = **26.00 dBi**
- $P_{tx} = 10 \log_{10}(3000)$ = **34.771 dBW**
- Thermal $kTB$ at 290 K, 500 Hz = **-176.985 dBW**
- Galactic $F_a = 52 - 23 \log_{10}(f_{MHz})$ at 31.5 MHz = **17.54 dB**
- Noise floor $N = kTB \cdot \max(NF, F_a)$ = **-159.447 dBW**, equivalent to
  **75.45 nV** at the 50 ohm input.


---
The next four chapters use this module unchanged. They differ only in
which transmitter and which propagation model is applied.
